# KnowWhere vs Lexical Baseline — A/B Evaluation

This notebook evaluates the KnowWhere retrieval system against a pure lexical baseline
(BM25) on a random sample of 1,000 queries from the AutoScholarQuery dataset.

## Structure
1. **Data Preparation** — Sample queries, extract ground-truth arXiv IDs
2. **Ingestion** — Fetch and ingest ground-truth papers via arXiv API
3. **Lexical Run** — BM25-only retrieval (mode='lexical')
4. **Hybrid Run** — Hybrid vector + keyword + reranker (mode='hybrid')
5. **Metrics** — Compute Precision@k, Recall@k, MRR, Latency

## 1. Data Preparation

Load the dataset, randomly sample 1,000 queries, and extract the unique arXiv IDs
that are listed as ground-truth answers.

In [1]:
import json
import random
import sys
from pathlib import Path

# Resolve project root from cwd
_ROOT = Path.cwd().resolve()
while not (_ROOT / ".env").exists():
    if _ROOT == _ROOT.parent:
        raise RuntimeError("Cannot find project root")
    _ROOT = _ROOT.parent

DATA_DIR = _ROOT / "eval" / "dataset"
OUTPUTS_DIR = _ROOT / "eval" / "outputs"
LIB_DIR = _ROOT / "eval" / "lib"
ENV_PATH = _ROOT / ".env"

OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(LIB_DIR))

SEED = 42
SAMPLE_SIZE = 1000

random.seed(SEED)
print(f"Project root: {_ROOT}")


Project root: /Users/bpmiranda3099/Documents/Projects/Academic/Thesis/knowwhere


In [2]:
# Load full dataset
train_path = DATA_DIR / "train.jsonl"
with open(train_path, "r") as f:
    all_queries = [json.loads(line) for line in f if line.strip()]

print(f"Total queries in dataset: {len(all_queries)}")

# Random sample
sample = random.sample(all_queries, SAMPLE_SIZE)
print(f"Sampled: {len(sample)}")

Total queries in dataset: 33551
Sampled: 1000


In [3]:
# Extract all unique arXiv IDs from ground-truth answers
all_arxiv_ids = set()
for query in sample:
    for aid in query.get("answer_arxiv_id", []):
        if aid:
            all_arxiv_ids.add(aid.strip())

print(f"Unique arXiv IDs in ground truth: {len(all_arxiv_ids)}")

# Stats
answer_counts = [len(q.get("answer_arxiv_id", [])) for q in sample]
print(f"Avg answers per query: {sum(answer_counts) / len(answer_counts):.2f}")
print(f"Min/Max answers per query: {min(answer_counts)}/{max(answer_counts)}")

Unique arXiv IDs in ground truth: 2188
Avg answers per query: 2.51
Min/Max answers per query: 1/14


In [4]:
# Save sampled queries for reproducibility
sample_path = OUTPUTS_DIR / "test_sample.jsonl"
with open(sample_path, "w") as f:
    for query in sample:
        f.write(json.dumps(query) + "\n")

# Save arXiv IDs list
arxiv_ids_path = OUTPUTS_DIR / "ingested_arxiv_ids.json"
with open(arxiv_ids_path, "w") as f:
    json.dump(sorted(all_arxiv_ids), f, indent=2)

print(f"Saved sample to: {sample_path}")
print(f"Saved arXiv IDs to: {arxiv_ids_path}")

Saved sample to: /Users/bpmiranda3099/Documents/Projects/Academic/Thesis/knowwhere/eval/outputs/test_sample.jsonl
Saved arXiv IDs to: /Users/bpmiranda3099/Documents/Projects/Academic/Thesis/knowwhere/eval/outputs/ingested_arxiv_ids.json


## 2. Ingestion

Ingest ground-truth arXiv papers into the KnowWhere database.
Papers are fetched from the arXiv API, embedded, chunked, and inserted into PostgreSQL.

**Skip this section on re-runs if papers are already ingested.**

In [5]:
from ingestion import ingest_from_titles, connect_db, get_existing_ids

# Build arXiv ID -> title mapping from the dataset
id_to_title: dict[str, str] = {}
for query in sample:
    for aid, title in zip(
        query.get("answer_arxiv_id", []),
        query.get("answer", []),
    ):
        if aid:
            id_to_title[aid.strip()] = title

conn = connect_db()
try:
    existing = get_existing_ids(conn, sorted(id_to_title.keys()))
    print(f"Already in DB: {len(existing)}")
    print(f"Need to ingest: {len(id_to_title) - len(existing)}")
    print(f"Total unique papers: {len(id_to_title)}")
finally:
    conn.close()


Already in DB: 2188
Need to ingest: 0
Total unique papers: 2188


In [6]:
# Create minimal paper entries from arXiv ID + title pairs.
# No arXiv API calls — uses local embedding service only.
# ~3-5 minutes for ~2,200 papers.

total_stats = ingest_from_titles(id_to_title, batch_size=100)

# Save ingestion stats
stats_path = OUTPUTS_DIR / "ingestion_stats.json"
with open(stats_path, "w") as f:
    json.dump(total_stats, f, indent=2)
print(f"Ingestion stats saved to: {stats_path}")


All papers already in DB.
Ingestion stats saved to: /Users/bpmiranda3099/Documents/Projects/Academic/Thesis/knowwhere/eval/outputs/ingestion_stats.json


## 3. Setup — Search Client & Metric Helpers

In [7]:
import requests
import time

from evaluation import evaluate_single_query

# Load config from project .env
ENV_PATH = _ROOT / ".env"
env_vars = {}
if ENV_PATH.exists():
    with open(ENV_PATH) as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                key, _, value = line.partition("=")
                env_vars[key.strip()] = value.strip()

API_PORT = env_vars.get("PORT", "3000")
API_URL = f"http://localhost:{API_PORT}/search"
API_KEY = env_vars.get("API_KEY", "")
HEADERS = {"Content-Type": "application/json", "x-api-key": API_KEY}
print(f"API_KEY SET" if API_KEY else "API_KEY NOT SET")

# Rate limit: 500 req/min = 120ms min; 150ms for safety
REQ_DELAY_S = 0.150

def search(query: str, mode: str, limit: int = 50) -> tuple[list[dict], float]:
    """
    Execute a single search query against the KnowWhere API.
    Includes rate-limit-aware delay before each request.

    Returns:
        Tuple of (results list, latency in milliseconds).
    """
    time.sleep(REQ_DELAY_S)
    payload = {"q": query, "mode": mode, "level": "paper", "limit": limit}
    start = time.perf_counter()
    resp = requests.post(API_URL, headers=HEADERS, json=payload, timeout=30)
    latency_ms = (time.perf_counter() - start) * 1000
    resp.raise_for_status()
    return resp.json().get("results", []), latency_ms


API_KEY SET


## 3a. Run — Lexical Mode (BM25)

Execute all 1,000 queries against KnowWhere in lexical-only mode.
Each result is matched against the ground-truth arXiv IDs.

In [8]:
MODE = "lexical"
RESULTS_LEXICAL = OUTPUTS_DIR / "results_lexical.jsonl"

with open(RESULTS_LEXICAL, "w") as outf:
    for i, query_data in enumerate(sample):
        qid = query_data["qid"]
        question = query_data["question"]
        ground_truth = set(query_data.get("answer_arxiv_id", []))

        try:
            results, latency = search(question, mode=MODE, limit=50)
            metrics = evaluate_single_query(results, ground_truth)
            metrics["latency_ms"] = latency

            record = {
                "qid": qid,
                "question": question,
                "latency_ms": latency,
                "num_results": len(results),
                "retrieved_ids": [r.get("id", "") for r in results],
                "ground_truth_ids": list(ground_truth),
                **metrics,
            }
        except Exception as exc:
            record = {
                "qid": qid,
                "question": question,
                "error": str(exc),
                "latency_ms": 0,
                "mrr": 0,
            }
            print(f"  Error on {qid}: {exc}")

        outf.write(json.dumps(record) + "\n")

        if (i + 1) % 100 == 0:
            print(f"Processed {i + 1}/{SAMPLE_SIZE} queries...")

print(f"Processed {SAMPLE_SIZE}/{SAMPLE_SIZE} queries...")
print(f"Results saved to: {RESULTS_LEXICAL}")

Processed 100/1000 queries...
Processed 200/1000 queries...
Processed 300/1000 queries...
Processed 400/1000 queries...
Processed 500/1000 queries...
Processed 600/1000 queries...
Processed 700/1000 queries...
Processed 800/1000 queries...
Processed 900/1000 queries...
Processed 1000/1000 queries...
Processed 1000/1000 queries...
Results saved to: /Users/bpmiranda3099/Documents/Projects/Academic/Thesis/knowwhere/eval/outputs/results_lexical.jsonl


In [9]:
# Load lexical results and compute summary
from evaluation import aggregate_metrics, format_results

lexical_metrics_list = []
lexical_latencies = []

with open(RESULTS_LEXICAL, "r") as f:
    for line in f:
        if not line.strip():
            continue
        rec = json.loads(line)
        if "error" in rec:
            continue
        lexical_metrics_list.append(rec)
        lexical_latencies.append(rec.get("latency_ms", 0))

lexical_summary = aggregate_metrics(lexical_metrics_list, lexical_latencies)
print(format_results("Lexical", lexical_summary))

--- FINAL EVALUATION RESULTS ---
Mode: Lexical
Total Queries Evaluated: 1000
Mean MRR: 0.0028
Mean Precision@5: 0.0011
Mean Precision@10: 0.0009
Mean Precision@20: 0.0008
Mean Precision@50: 0.0008
Mean Recall@5: 0.0010
Mean Recall@10: 0.0010
Mean Recall@20: 0.0010
Mean Recall@50: 0.0010
Average End-to-End Latency: 85.06 ms


## 3b. Run — Hybrid Mode (Vector + Keyword + Reranker)

Execute the same 1,000 queries in hybrid mode, which combines
BM25 lexical scoring with semantic vector similarity and cross-encoder reranking.

In [10]:
RESULTS_HYBRID = OUTPUTS_DIR / "results_hybrid.jsonl"

with open(RESULTS_HYBRID, "w") as outf:
    for i, query_data in enumerate(sample):
        qid = query_data["qid"]
        question = query_data["question"]
        ground_truth = set(query_data.get("answer_arxiv_id", []))

        try:
            results, latency = search(question, mode="hybrid", limit=50)
            metrics = evaluate_single_query(results, ground_truth)
            metrics["latency_ms"] = latency

            record = {
                "qid": qid,
                "question": question,
                "latency_ms": latency,
                "num_results": len(results),
                "retrieved_ids": [r.get("id", "") for r in results],
                "ground_truth_ids": list(ground_truth),
                **metrics,
            }
        except Exception as exc:
            record = {
                "qid": qid,
                "question": question,
                "error": str(exc),
                "latency_ms": 0,
                "mrr": 0,
            }
            print(f"  Error on {qid}: {exc}")

        outf.write(json.dumps(record) + "\n")

        if (i + 1) % 100 == 0:
            print(f"Processed {i + 1}/{SAMPLE_SIZE} queries...")

print(f"Processed {SAMPLE_SIZE}/{SAMPLE_SIZE} queries...")
print(f"Results saved to: {RESULTS_HYBRID}")

Processed 100/1000 queries...
Processed 200/1000 queries...
Processed 300/1000 queries...
Processed 400/1000 queries...
Processed 500/1000 queries...
Processed 600/1000 queries...
Processed 700/1000 queries...
Processed 800/1000 queries...
Processed 900/1000 queries...
Processed 1000/1000 queries...
Processed 1000/1000 queries...
Results saved to: /Users/bpmiranda3099/Documents/Projects/Academic/Thesis/knowwhere/eval/outputs/results_hybrid.jsonl


In [11]:
# Load hybrid results and compute summary
hybrid_metrics_list = []
hybrid_latencies = []

with open(RESULTS_HYBRID, "r") as f:
    for line in f:
        if not line.strip():
            continue
        rec = json.loads(line)
        if "error" in rec:
            continue
        hybrid_metrics_list.append(rec)
        hybrid_latencies.append(rec.get("latency_ms", 0))

hybrid_summary = aggregate_metrics(hybrid_metrics_list, hybrid_latencies)
print(format_results("Hybrid", hybrid_summary))

--- FINAL EVALUATION RESULTS ---
Mode: Hybrid
Total Queries Evaluated: 1000
Mean MRR: 0.3619
Mean Precision@5: 0.1390
Mean Precision@10: 0.0945
Mean Precision@20: 0.0657
Mean Precision@50: 0.0346
Mean Recall@5: 0.3181
Mean Recall@10: 0.4131
Mean Recall@20: 0.5569
Mean Recall@50: 0.6704
Average End-to-End Latency: 1315.68 ms


## 4. Summary & Side-by-Side Comparison

In [12]:
import numpy as np

def compare_modes(lex: dict, hyb: dict):
    """Print a side-by-side comparison table with percentage deltas."""
    print(f"{'Metric':<30} {'Lexical':>12} {'Hybrid':>12} {'Delta':>12}")
    print("-" * 68)

    metrics = ["mean_mrr"]
    for k in (5, 10, 20, 50):
        metrics.append(f"mean_precision@{k}")
    for k in (5, 10, 20, 50):
        metrics.append(f"mean_recall@{k}")

    for metric in metrics:
        lex_val = lex.get(metric, 0)
        hyb_val = hyb.get(metric, 0)
        delta = (hyb_val - lex_val) / lex_val * 100 if lex_val > 0 else float("inf")
        name = metric.replace("mean_", "")
        print(f"{name:<30} {lex_val:>12.4f} {hyb_val:>12.4f} {delta:>+11.1f}%")

    # Latency
    lex_lat = lex.get("mean_latency_ms", 0)
    hyb_lat = hyb.get("mean_latency_ms", 0)
    if lex_lat > 0:
        lat_delta = (hyb_lat - lex_lat) / lex_lat * 100
        print(f"{'latency_ms':<30} {lex_lat:>12.2f} {hyb_lat:>12.2f} {lat_delta:>+11.1f}%")

compare_modes(lexical_summary, hybrid_summary)

Metric                              Lexical       Hybrid        Delta
--------------------------------------------------------------------
mrr                                  0.0028       0.3619    +12672.0%
precision@5                          0.0011       0.1390    +12931.2%
precision@10                         0.0009       0.0945    +10665.8%
precision@20                         0.0008       0.0657     +7836.9%
precision@50                         0.0008       0.0346     +4116.9%
recall@5                             0.0010       0.3181    +30849.6%
recall@10                            0.0010       0.4131    +40090.1%
recall@20                            0.0010       0.5569    +54086.5%
recall@50                            0.0010       0.6704    +65131.1%
latency_ms                            85.06      1315.68     +1446.7%


In [13]:
# Save final metrics to JSON
final_metrics = {
    "lexical": lexical_summary,
    "hybrid": hybrid_summary,
    "sample_size": SAMPLE_SIZE,
    "seed": SEED,
}

metrics_path = OUTPUTS_DIR / "metrics.json"
with open(metrics_path, "w") as f:
    json.dump(final_metrics, f, indent=2)

print(f"Final metrics saved to: {metrics_path}")

Final metrics saved to: /Users/bpmiranda3099/Documents/Projects/Academic/Thesis/knowwhere/eval/outputs/metrics.json
